In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict  
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import Annotated
import operator
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite"
)

In [3]:
class jokeState(TypedDict):
    joke : str
    explanation : str
    topic : str

In [9]:
def generate_joke(state : jokeState):
    prompt = f'''generate a joke on the topic\n {state['topic']}'''

    response = model.invoke(prompt).content
    return {
        'joke' : response
    }

def generate_explanation(state : jokeState):
    prompt = f'''generate explanation of the following joke\n {state['joke']}'''

    response = model.invoke(prompt).content
    return {
        'explanation' : response
    }

In [10]:
graph = StateGraph(jokeState)
checkpointer = InMemorySaver()

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {'configurable' : {'thread_id' : '1'}}

response = workflow.invoke({
    'topic' : 'relationships'
}, config1)

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'joke': [{'type': 'text', 'text': "My wife told me that sex is much better on vacation. \n\nThat wasn't a very nice postcard to receive.", 'extras': {'signature': 'EmAKXgFpFH0TLJHQ/hqRn6+9qwL8E9klwlh1XKPO2B+LzXQzhwqXymBfb2nUzYof2FzPpNXq5YxWfqCDc5jM/u8Oe+O6FcXOp9uU9X8OgdH9bP1w5K9iQjdEMSatvp317/Y='}}], 'explanation': [{'type': 'text', 'text': "This joke relies on a **double meaning** and a **surprise twist** regarding how the husband receives a message from his wife. \n\nHere is the breakdown of why it is funny:\n\n1. **The Setup:** *“My wife told me that sex is much better on vacation.”*\n   * When you first read this, you assume the wife is telling her husband this directly—either while they are talking at home, or perhaps as a piece of relationship feedback. \n\n2. **The Punchline:** *“That wasn't a very nice postcard to receive.”*\n   * This completely flips the context. The husband is not with his wife; **she is on vacation *without* him**, and she sent him a 

In [15]:
response = workflow.invoke({
    'topic' : 'teachers'
}, config1)

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'joke': [{'type': 'text', 'text': 'Teacher: "If I gave you 2 cats and another 2 cats, how many cats would you have?"\n\nJohnny: "Five, ma\'am."\n\nTeacher (sighing): "Let\'s try again. If I gave you 2 *apples* and another 2 *apples*, how many apples would you have?"\n\nJohnny: "Four."\n\nTeacher: "Good. Now, if I gave you 2 *cats* and another 2 *cats*, how many cats would you have?"\n\nJohnny: "Five!"\n\nTeacher (frustrated): "How on earth do you get five?!"\n\nJohnny: "Because I already have one cat at home!"', 'extras': {'signature': 'EmAKXgFpFH0T8NLjzy4HDBUE6QXFflJMFToomqKuGhXt9Z7v+a68g7osEmr3kWqWmG2lKtrCL+EzRViUmCZ+FbY1VgXNVzjmy8lZbg6C2Eij+HOkwjGVys5PUf/mRWjhZQ4='}}], 'explanation': [{'type': 'text', 'text': 'Here is an explanation of the joke:\n\n**The Setup:**\nThe teacher is trying to use a simple math word problem (addition: $2 + 2$) using cats, expecting the logical answer of **4**. \n\n**The Misunderstanding:**\nWhen Johnny answers **"Five,"** the teach